In [ ]:
import pandas as pd
import requests
from io import StringIO

# Importint and Merging Data
This notebook contains the first importing, cleaning, and merging steps of the final assignment exercise.

## ISTAT Data

We initially load and inspect the ISTAT data.

In [ ]:
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

# as API to return a CSV rather than XML
headers = {
  "Accept": "application/vnd.sdmx.data+csv;version=1.0.0"
}

response = requests.get(url, headers=headers)
response.status_code


In [ ]:
type(response)
response.text
# As the data is saved in text, we use StringIO to turn it into a pandas object
df_istat = pd.read_csv(StringIO(response.text))

# Do some exporatory first checks on the data
df_istat.shape
df_istat.info()


<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

In [ ]:
df_istat.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Now that I have loaded the data I will do some checks to verify what is in the categorical values.

In [ ]:
df_istat["FREQ"].value_counts()

FREQ
A    573552
Name: count, dtype: int64

In [ ]:
df_istat["DATA_TYPE"].value_counts()

DATA_TYPE
KILLINJ    382368
ROADACC    191184
Name: count, dtype: int64

In [ ]:
df_istat["RESULT"].value_counts()

RESULT
F    191184
M    191184
9    191184
Name: count, dtype: int64

In [ ]:
df_istat.groupby(["DATA_TYPE", "RESULT"]).size()

DATA_TYPE  RESULT
KILLINJ    F         191184
           M         191184
ROADACC    9         191184
dtype: int64

In [ ]:
df_istat["TIME_PERIOD"].min()


np.int64(2001)

In [ ]:
df_istat["TIME_PERIOD"].max()

np.int64(2024)

In [ ]:
df_istat["REF_AREA"].nunique()

8578

The above tells us that the data is annual data per comune where for each combination comune-year it records the amount of people injured or dead, and the number of accidents.

Digging deeper online on the metadata I find that f = feriti, and m = morti, whereas 9 is simply an identifier for road-accident observations. Here is my reference: https://ondata.github.io/guida-api-istat/?utm_source=chatgpt.com.

I move on below to clean a bit the data by removing all the empty column. I am also only interested in recent year data so I limit myself to 2020 - 2024 period.

In [ ]:
df_istat = df_istat.drop(columns=[
    "OBS_STATUS",
    "NOTE_DS",
    "NOTE_REF_AREA",
    "NOTE_DATA_TYPE",
    "NOTE_RESULT",
    "NOTE_TIME_PERIOD",
    "BASE_PER",
    "UNIT_MEAS",
    "UNIT_MULT"
])



In [ ]:
df_istat = df_istat[df_istat["TIME_PERIOD"] >= 2020]
df_istat["TIME_PERIOD"].value_counts()

TIME_PERIOD
2020    23712
2021    23712
2022    23712
2023    23703
2024    19017
Name: count, dtype: int64

I note here that the numbers per year are different so not every municipality has numbers for every year.
To simplify the analysis later, we create proper labels and go from long to wide format by using pivot table.

In [ ]:
df_istat["MEASURE"] = ""

df_istat.loc[df_istat["DATA_TYPE"] == "ROADACC", "MEASURE"] = "ACCIDENTS"

df_istat.loc[
    (df_istat["DATA_TYPE"] == "KILLINJ") & 
    (df_istat["RESULT"] == "F"), 
    "MEASURE"
] = "INJURED"

df_istat.loc[
    (df_istat["DATA_TYPE"] == "KILLINJ") & 
    (df_istat["RESULT"] == "M"), 
    "MEASURE"
] = "KILLED"
df_istat.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,MEASURE
19,IT1:41_983(1.0),A,1001,KILLINJ,F,2020,4,INJURED
20,IT1:41_983(1.0),A,1001,KILLINJ,F,2021,2,INJURED
21,IT1:41_983(1.0),A,1001,KILLINJ,F,2022,5,INJURED
22,IT1:41_983(1.0),A,1001,KILLINJ,F,2023,6,INJURED
23,IT1:41_983(1.0),A,1001,KILLINJ,F,2024,5,INJURED


In [ ]:
df_istat["MEASURE"].value_counts()

MEASURE
INJURED      37952
KILLED       37952
ACCIDENTS    37952
Name: count, dtype: int64

In [ ]:
df_accidents = df_istat.pivot_table(
    index=["REF_AREA", "TIME_PERIOD"],
    columns="MEASURE",
    values="OBS_VALUE"
).reset_index()
df_accidents.columns.name = None
df_accidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 37952 entries, 0 to 37951
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   REF_AREA     37952 non-null  int64  
 1   TIME_PERIOD  37952 non-null  int64  
 2   ACCIDENTS    37952 non-null  float64
 3   INJURED      37952 non-null  float64
 4   KILLED       37952 non-null  float64
dtypes: float64(3), int64(2)
memory usage: 1.4 MB


In [ ]:
print(df_accidents.shape)
print(df_accidents.isna().sum())
print(df_accidents.duplicated(subset=["REF_AREA", "TIME_PERIOD"]).sum())

(37952, 5)
REF_AREA       0
TIME_PERIOD    0
ACCIDENTS      0
INJURED        0
KILLED         0
dtype: int64
0


## Merging with SITUAS data

Now we bring in the SITUAS data and do a similar procedure.

In [ ]:
import os
import time

from selenium import webdriver
from selenium.webdriver.common.by import By

download_folder = os.path.abspath("../data")
print(download_folder)

/Users/nicolascaballero/Documents/Personal/Education/Boolean/capstone_project/data


In [ ]:
# Step 1: Configure Chrome to make sure that the files save where they need to
chrome_options = webdriver.ChromeOptions()

prefs = {
    "download.default_directory": download_folder,
    # the below avoids a pop-up where chrome asks if im sure
    "download.prompt_for_download": False
}

# This attaches the configuration dictionary above to the chrome options instance.
chrome_options.add_experimental_option("prefs", prefs)

# This tells Selenium to launch Chrome using the settings I just created.
driver = webdriver.Chrome(options=chrome_options)

# Step 2: Loop through the 5 years of analysis and iteratively change the url that gets fed to the driver.
years = [2020, 2021, 2022, 2023, 2024]

for year in years:

    url = f"https://situas.istat.it/web/#/territorio/body?id=74&dateFrom={year}-12-31"

    driver.get(url)
    # This is to make sure python stops so the process loads. I found that it's necessary when debugging
    time.sleep(5)

    # Find the export button and click it
    export_button = driver.find_element(
        By.ID,
        "dati-report-export-btn"
    )
    export_button.click()
    # wait a bit
    time.sleep(1)

    # find the csv button and click it
    csv_button = driver.find_element(
       # The above is essentially "find button anywhere on the page where the aria-label equals 'Scarica dati in formato CSV' "
        By.XPATH,
        '//button[@aria-label="Scarica dati in formato CSV"]'
    )
    csv_button.click()
    time.sleep(3)

    # Step 3: Rename the file to something that we can then consistently call (without timestamp of when it was downloaded)

    # for every file in the list of files in the folder...
    for file in os.listdir(download_folder):
    # if the file name includes the year we are considering (nested for loop)
      if f"31-12-{year}" in file:
          # get the absolute path for the file
          old_name = os.path.join(download_folder, file)

          # create a new absolute path which replaces the file name
          new_name = os.path.join(
              download_folder,
              f"situas_{year}.csv"
          )

          # rename the file in the OS, from old_name to new_name
          os.rename(old_name, new_name)
    # Print in console when the whole process finishes
    print(f"Downloaded SITUAS data for {year}")

driver.quit()

Downloaded SITUAS data for 2020
Downloaded SITUAS data for 2021
Downloaded SITUAS data for 2022
Downloaded SITUAS data for 2023
Downloaded SITUAS data for 2024


In [ ]:
df_situas_2020 = pd.read_csv(
    "../data/situas_2020.csv",
    sep=";",
    decimal=",",
    thousands="."
)

df_situas_2021 = pd.read_csv(
    "../data/situas_2021.csv",
    sep=";",
    decimal=",",
    thousands="."
)

df_situas_2022 = pd.read_csv(
    "../data/situas_2022.csv",
    sep=";",
    decimal=",",
    thousands="."
)

df_situas_2023 = pd.read_csv(
    "../data/situas_2023.csv",
    sep=";",
    decimal=",",
    thousands="."
)

df_situas_2024 = pd.read_csv(
    "../data/situas_2024.csv",
    sep=";",
    decimal=",",
    thousands="."
)

In [ ]:
df_situas_2020["YEAR"] = 2020
df_situas_2021["YEAR"] = 2021
df_situas_2022["YEAR"] = 2022
df_situas_2023["YEAR"] = 2023
df_situas_2024["YEAR"] = 2024

In [ ]:
# Concatenate into one dataset
df_situas = pd.concat([
    df_situas_2020,
    df_situas_2021,
    df_situas_2022,
    df_situas_2023,
    df_situas_2024
], ignore_index=True)

In [ ]:
print(df_situas.shape)
print(df_situas["YEAR"].value_counts())

print(df_situas.columns)
# chanve in number of observations could be municipality boundaries changing over time?

(39507, 18)
YEAR
2021    7904
2022    7904
2020    7903
2023    7900
2024    7896
Name: count, dtype: int64
Index(['Codice Ripartizione geografica', 'Codice Regione',
       'Codice Provincia (Storico)', 'Codice Provincia/Uts',
       'Codice Comune (alfanumerico)', 'Codice Comune (numerico)', 'Comune',
       'Comune (dizione straniera)', 'Sigla automobilistica',
       'Capoluogo di Provincia/Uts', 'Capoluogo di Regione',
       'Popolazione legale', 'Anno Censimento', 'Superficie (Kmq)',
       'Anno (Superficie)', 'Popolazione residente',
       'Anno (Popolazione residente)', 'YEAR'],
      dtype='str')


Now that we have imported and concatenated the data, we will do some cleaning and retain only important variables for analysis, then perform the merge with the cleaned ISTAT accident data.

In [ ]:
# Select only variables we want
df_situas = df_situas[[
    "Codice Regione",
    "Codice Provincia/Uts",
    "Codice Comune (numerico)",
    "Comune",
    "Sigla automobilistica",
    "Superficie (Kmq)",
    "Popolazione residente",
    "YEAR"
]]

# rename into something easier to work with
df_situas = df_situas.rename(columns={
    "Codice Regione": "REGION_CODE",
    "Codice Provincia/Uts": "PROVINCE_CODE",
    "Codice Comune (numerico)": "REF_AREA",
    "Comune": "MUNICIPALITY",
    "Sigla automobilistica": "PROVINCE",
    "Superficie (Kmq)": "AREA_KM2",
    "Popolazione residente": "POPULATION"
})

In [ ]:
# Check that the two merging variables have compatible types
print(df_accidents[["REF_AREA", "TIME_PERIOD"]].dtypes)
print(df_situas[["REF_AREA", "YEAR"]].dtypes)

NameError: name 'df_accidents' is not defined

## Merge, clean final dataset, and generate indicators for analysis

In [ ]:
df_final = df_accidents.merge(
    df_situas,
    left_on=["REF_AREA", "TIME_PERIOD"],
    right_on=["REF_AREA", "YEAR"],
    how="left"
)
df_final.head(10)

In [43]:
# Check that the merge happened as expected
print(df_final.shape)
print(df_final.isna().sum())

df_final.head()

(37952, 12)
REF_AREA           0
TIME_PERIOD        0
ACCIDENTS          0
INJURED            0
KILLED             0
REGION_CODE        2
PROVINCE_CODE      2
MUNICIPALITY       7
PROVINCE         461
AREA_KM2           2
POPULATION         2
YEAR               2
dtype: int64


,REF_AREA,TIME_PERIOD,ACCIDENTS,INJURED,KILLED,REGION_CODE,PROVINCE_CODE,MUNICIPALITY,PROVINCE,AREA_KM2,POPULATION,YEAR
0,1001,2020,4.0,4.0,0.0,1.0,201.0,Agliè,TO,13.1462,2545.0,2020.0
1,1001,2021,2.0,2.0,0.0,1.0,201.0,Agliè,TO,13.1463,2562.0,2021.0
2,1001,2022,3.0,5.0,0.0,1.0,201.0,Agliè,TO,13.1463,2568.0,2022.0
3,1001,2023,3.0,6.0,1.0,1.0,201.0,Agliè,TO,13.1463,2596.0,2023.0
4,1001,2024,2.0,5.0,0.0,1.0,201.0,Agliè,TO,13.1463,2585.0,2024.0


Here I stop a second because I see that there are some missings in variables like municipality and I want to investigate why.
As REGION_CODE only has two missings, the merge essentially worked except for those two.
However the fact that SITUAS variables are missing in some informative labels tells us some region labels in SITUAS data are not necessarily associated to a municipality/province name.

In [44]:
# Identify rows that failed merge
df_final[df_final["POPULATION"].isna()]

,REF_AREA,TIME_PERIOD,ACCIDENTS,INJURED,KILLED,REGION_CODE,PROVINCE_CODE,MUNICIPALITY,PROVINCE,AREA_KM2,POPULATION,YEAR
10868,18002,2023,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19189,41032,2020,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
df_final[df_final["MUNICIPALITY"].isna()]

,REF_AREA,TIME_PERIOD,ACCIDENTS,INJURED,KILLED,REGION_CODE,PROVINCE_CODE,MUNICIPALITY,PROVINCE,AREA_KM2,POPULATION,YEAR
776,1168,2020,8.0,11.0,0.0,1.0,201.0,NaN,TO,24.6422,7849.0,2020.0
777,1168,2021,9.0,11.0,0.0,1.0,201.0,NaN,TO,24.6422,7830.0,2021.0
778,1168,2022,7.0,10.0,0.0,1.0,201.0,NaN,TO,24.6422,7790.0,2022.0
779,1168,2023,7.0,9.0,0.0,1.0,201.0,NaN,TO,24.6422,7688.0,2023.0
780,1168,2024,5.0,7.0,0.0,1.0,201.0,NaN,TO,24.6422,7662.0,2024.0
10868,18002,2023,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19189,41032,2020,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_final[df_final["PROVINCE"].isna()][
    ["REF_AREA", "TIME_PERIOD", "MUNICIPALITY", "REGION_CODE", "PROVINCE_CODE", "PROVINCE"]
].head(20)

,REF_AREA,TIME_PERIOD,MUNICIPALITY,REGION_CODE,PROVINCE_CODE
10868,18002,2023,NaN,NaN,NaN
19189,41032,2020,NaN,NaN,NaN
24418,63001,2020,Acerra,15.0,263.0
24419,63001,2021,Acerra,15.0,263.0
24420,63001,2022,Acerra,15.0,263.0
24421,63001,2023,Acerra,15.0,263.0
24422,63001,2024,Acerra,15.0,263.0
24423,63002,2020,Afragola,15.0,263.0
24424,63002,2021,Afragola,15.0,263.0
24425,63002,2022,Afragola,15.0,263.0


Below we deal with these edge cases where we have missings.

Two accident observations could not be matched to SITUAS and were excluded because population and surface area information were unavailable; both observations recorded zero accidents. 

Some observations with missing municipality and province labels from SITUAS that were genuniely missing were retained where the underlying population and surface-area data were available, as these do not prevent calculation of the required indicators.

In [ ]:
# Drop the variables that genuinely failed to merge (perhaps administrative area changes?)
# Only drops in NA in the subset population.
df_final = df_final.dropna(subset=["POPULATION"])

In [48]:
df_situas[
    (df_situas["REF_AREA"] >= 1165) &
    (df_situas["REF_AREA"] <= 1170)
][["REF_AREA", "MUNICIPALITY"]].drop_duplicates()

,REF_AREA,MUNICIPALITY
161,1165,Noasca
162,1166,Nole
163,1167,Nomaglio
164,1168,NaN
165,1169,Novalesa
166,1170,Oglianico


Before moving on to creating indicators we quickly look at extreme values or outliers.

In [51]:
# Check distributions of the main variables
print(df_final[[
    "ACCIDENTS",
    "INJURED",
    "KILLED",
    "POPULATION",
    "AREA_KM2"
]].describe())


# Check for impossible negative values
print("Negative accidents:", (df_final["ACCIDENTS"] < 0).sum())
print("Negative injured:", (df_final["INJURED"] < 0).sum())
print("Negative killed:", (df_final["KILLED"] < 0).sum())
print("Negative population:", (df_final["POPULATION"] < 0).sum())
print("Negative area:", (df_final["AREA_KM2"] < 0).sum())

          ACCIDENTS       INJURED        KILLED    POPULATION      AREA_KM2
count  37950.000000  37950.000000  37950.000000  3.795000e+04  37950.000000
mean      20.446667     27.560949      0.382029  7.744012e+03     38.808091
std      184.483342    232.057227      1.960904  4.244042e+04     51.537956
min        0.000000      0.000000      0.000000  2.900000e+01      0.120600
25%        1.000000      1.000000      0.000000  1.064000e+03     11.627100
50%        3.000000      5.000000      0.000000  2.539000e+03     22.707200
75%       11.000000     16.000000      0.000000  6.491750e+03     45.225500
max    13924.000000  17196.000000    154.000000  2.770226e+06   1288.187500
Negative accidents: 0
Negative injured: 0
Negative killed: 0
Negative population: 0
Negative area: 0


In [52]:
# Municipalities with the highest number of accidents
print(df_final.nlargest(10, "ACCIDENTS")[
    ["MUNICIPALITY", "TIME_PERIOD", "ACCIDENTS", "POPULATION"]
])

# Municipalities with the smallest populations
print(df_final.nsmallest(10, "POPULATION")[
    ["MUNICIPALITY", "TIME_PERIOD", "ACCIDENTS", "POPULATION"]
])

# Municipalities with the smallest areas
print(df_final.nsmallest(10, "AREA_KM2")[
    ["MUNICIPALITY", "TIME_PERIOD", "ACCIDENTS", "AREA_KM2"]
])

      MUNICIPALITY  TIME_PERIOD  ACCIDENTS  POPULATION
22807         Roma         2024    13924.0   2747290.0
22805         Roma         2022    13181.0   2755309.0
22806         Roma         2023    12817.0   2751747.0
22804         Roma         2021    11129.0   2749031.0
22803         Roma         2020     8229.0   2770226.0
8373        Milano         2023     7812.0   1371499.0
8372        Milano         2022     7783.0   1358420.0
8374        Milano         2024     7743.0   1365698.0
8371        Milano         2021     7465.0   1349930.0
8370        Milano         2020     4856.0   1374582.0
      MUNICIPALITY  TIME_PERIOD  ACCIDENTS  POPULATION
35604    Morterone         2020        0.0        29.0
35605    Morterone         2021        0.0        31.0
35606    Morterone         2022        0.0        32.0
35607    Morterone         2023        1.0        33.0
7863      Pedesina         2021        0.0        35.0
7865      Pedesina         2023        0.0        35.0
7862      

Extreme values: Descriptive statistics and the most extreme observations were inspected for accident counts, population and municipality area. Accident counts are strongly right-skewed, with the largest values corresponding to highly populated municipalities such as Roma and Milano. Very small population and area values were also identified, but these correspond to small municipalities rather than apparent data errors. Therefore, extreme observations were retained.

Now we move on to creating indicators, and checking some values for them too.

In [53]:
# Create indicators
df_final["ACCIDENTS_PER_CAPITA"] = (
    df_final["ACCIDENTS"] / df_final["POPULATION"]
)

df_final["ACCIDENTS_PER_KM2"] = (
    df_final["ACCIDENTS"] / df_final["AREA_KM2"]
)

# Check indicator distributions
print(df_final[[
    "ACCIDENTS_PER_CAPITA",
    "ACCIDENTS_PER_KM2"
]].describe())

       ACCIDENTS_PER_CAPITA  ACCIDENTS_PER_KM2
count          37950.000000       37950.000000
mean               0.001746           0.598106
std                0.001864           1.553808
min                0.000000           0.000000
25%                0.000576           0.031960
50%                0.001419           0.149727
75%                0.002415           0.505414
max                0.085106          42.959663


In [54]:
# Investigate extreme values of the created indicators
# Highest accidents per capita
print(df_final.nlargest(10, "ACCIDENTS_PER_CAPITA")[
    ["MUNICIPALITY", "TIME_PERIOD", "ACCIDENTS",
     "POPULATION", "ACCIDENTS_PER_CAPITA"]
])

# Highest accidents per km2
print(df_final.nlargest(10, "ACCIDENTS_PER_KM2")[
    ["MUNICIPALITY", "TIME_PERIOD", "ACCIDENTS",
     "AREA_KM2", "ACCIDENTS_PER_KM2"]
])

             MUNICIPALITY  TIME_PERIOD  ACCIDENTS  POPULATION  \
727            Moncenisio         2022        4.0        47.0   
2310            Argentera         2024        3.0        77.0   
35607           Morterone         2023        1.0        33.0   
4848                 Bard         2023        3.0       102.0   
35311  Villanova Biellese         2023        5.0       175.0   
3037         Pietraporzio         2020        2.0        74.0   
15085   Colle Santa Lucia         2024        9.0       340.0   
6161            Carrodano         2023       12.0       465.0   
17626           Cerignale         2021        3.0       117.0   
34656            Baradili         2020        2.0        80.0   

       ACCIDENTS_PER_CAPITA  
727                0.085106  
2310               0.038961  
35607              0.030303  
4848               0.029412  
35311              0.028571  
3037               0.027027  
15085              0.026471  
6161               0.025806  
17626         

Indicator extreme-value checks: Extreme values in the derived indicators were investigated. High accidents-per-capita values were primarily associated with municipalities with very small populations, where a small number of accidents produces a high per-capita rate. High accidents-per-km² values were associated with dense urban municipalities. As these values are consistent with the underlying population, area and accident data, they were retained.

In [55]:
# Save cleaned data to take in the other notebook
df_final.to_csv("../data/df_final.csv", index=False)